# Solar filament instance segmentation: demo

The Kaggle competition [filament-segmentation-2026](https://www.kaggle.com/competitions/filament-segmentation-2026)
asks for a separate mask for every solar filament in 2048 x 2048 GONG H-alpha full-disk images. A submission is a CSV
of masks in COCO run-length encoding (RLE, a compact way to store a mask as runs of pixels). Kaggle scores it with
panoptic quality (PQ), pooled over every annotator's drawing of every test image. A predicted mask only matches a
drawn filament when their IoU (intersection over union, the shared area divided by the combined area) is above 0.5.

I started with the simplest idea: threshold a U-Net probability map and take each connected blob as one filament,
which gave a development PQ of 0.383. One threshold merges neighboring filaments or splits long ones, though, and
annotators disagree about where filaments end. So instead of picking one threshold, the pipeline makes many candidate
masks, trains a model to predict how well each candidate will match the annotators, and picks the best set of
non-overlapping candidates exactly, which raised development PQ to 0.438. Next I gave that scorer more evidence,
namely what other models (a detector-refiner, Mask R-CNN and two more U-Nets) think of each candidate, and development
PQ went to about 0.46. Last, I refined the decision itself with safer thresholds, a blend of two kinds of scorer and a
0.23 px nudge of the masks toward the top of the image, which reached 0.4706 on my development images (an optimistic estimate, since I made my
choices on them) and 0.41 on the public leaderboard.

In concrete terms, the pipeline proposes candidate masks from three U-Net probability maps, a Mask R-CNN detector and
the I6 detector-refiner. It predicts the expected IoU of each candidate with LightGBM and ExtraTrees models, picks a
non-overlapping set for each image with an integer program, and writes the masks. This notebook runs it one stage at a
time and rebuilds the evaluated submission `s1-S-t024.csv` (Kaggle public score 0.41).

| # | What the stage does | Module | Runs on |
|---|---|---|---|
| 1 | Reads the 180 test images. | `filseg/data.py` | CPU |
| 2 | Three U-Nets (cnx, cnxL and R34, each with 4 fold copies) make probability maps. | `filseg/dense.py` | GPU, or CPU |
| 3 | Mask R-CNN (4 fold copies) proposes its own filament masks. | `filseg/det.py` | GPU, or CPU |
| 4 | The I6 detector finds filament regions and its refiner outlines them. | `filseg/i6.py` | GPU, or CPU |
| 5 | The code builds the candidate masks and works out 86 evidence numbers for each one. | `filseg/evidence.py`, `filseg/rows.py` | CPU |
| 6 | The scorers predict each candidate's expected IoU, q. | `filseg/score.py` | CPU |
| 7 | An exact integer program picks the best set of candidates that don't overlap. | `filseg/select.py` | CPU |
| 8 | The picked masks move 0.23 px toward the top of the image, and overlapping pixels go to the higher score. | `filseg/shift.py` | CPU |
| 9 | The code writes the submission CSV. | `filseg/write.py` | CPU |
| 10 | The code checks the CSV. | `filseg/validate.py` | CPU |

Each stage below is one call to `filseg/pipeline.py`, and `pipeline.run(mode=..., device=...)` runs the same chain in
one call.

## Setup

This cell sets the folders and the run mode, imports the pipeline, and checks that the test images are there.

- `DATA` is the unzipped competition folder, the one that holds `test/test_images` (and `train/` for the evaluation
  section). `WORK` is a writable folder for intermediate files, which take several GB in full mode.
- `MODE = 'replay'` skips stages 1-4. It unpacks `replay/`, which holds the stored stage 2-4 outputs of the run that
  produced the submission, checks every file's sha256, and runs stages 5-10 on them. The result is byte-identical to
  the submitted file. It runs on the CPU and took about 17 minutes on a desktop CPU with 4 worker processes. Replay mode writes
  about 4.5 GB to `WORK`.
- `MODE = 'full'` runs every stage from the test images. Stages 2-4 run on `DEVICE`, and on one RTX 3070 Ti the whole
  run took about 83 minutes. GPUs differ slightly in floating-point arithmetic, and the U-Nets and I6 run in bfloat16,
  so the result is a close copy of the submission rather than the same bytes.
- `DEVICE` defaults to `'cpu'`, so the default run never touches a GPU. Set `DEVICE = 'cuda'` for full mode, because
  stages 2-4 take many hours on a CPU.
- `TRAIN = True` turns on the optional retraining section at the end. It is off by default, and the pipeline uses the
  trained weights shipped in `weights/`.

In [ ]:
import os
import sys
from pathlib import Path

REPO = Path.cwd()                      # run the notebook from the repository root
DATA = Path(os.environ.get('FILSEG_DATA', REPO / 'data' / 'MAGFiLO_1.0_Kaggle_2026'))
WORK = Path(os.environ.get('FILSEG_WORK', REPO / 'work'))

MODE = 'replay'      # 'replay' rebuilds the submission exactly on the CPU from stored stage 2-4 outputs, 'full' runs every stage
DEVICE = 'cpu'       # stages 2-4 of full mode and the evaluation section run here. Set this to 'cuda' for full mode.
TRAIN = False        # True turns on the optional retraining section at the end
NPROC = 4            # number of worker processes for stage 5

if DEVICE == 'cpu':
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'   # hide the GPUs before torch is imported (an empty string doesn't hide them on Windows)
os.environ['FILSEG_DATA'], os.environ['FILSEG_WORK'] = str(DATA), str(WORK)
sys.path.insert(0, str(REPO))

from filseg import pipeline

IMAGES = pipeline.test_images(DATA)
WORK_RUN = WORK / f'run-{MODE}'        # each mode gets its own work folder
OUT_CSV = WORK_RUN / 'submission.csv'
assert IMAGES.is_dir(), f'test images not found: {IMAGES}'
print(f'mode {MODE} | device {DEVICE} | {len(os.listdir(IMAGES))} test images in {IMAGES} | work {WORK_RUN}')

## Stage 1: test images (full mode)

This cell reads the 180 test images, because every later stage works from them and the U-Nets need the brightness of
each image's solar disk. It reads the JPEGs in sorted file-name order as 2048 x 2048 grayscale arrays. For each image
it computes the mean and standard deviation of the solar-disk pixels, meaning the pixels above 30 % of the 99th
percentile, and the U-Nets normalize their input with these two numbers. The arrays are cached in `WORK_RUN/cache`.

In [ ]:
if MODE == 'full':
    recs, images = pipeline.stage1_images(IMAGES, WORK_RUN)
else:
    print('replay mode: stages 1-4 come from replay/ (cell after stage 4)')

## Stage 2: U-Net probability maps (full mode)

This cell runs three U-Nets over every image to get maps of how likely each pixel is to be part of a filament, and
these maps are the raw material for nearly every candidate. The cnx mask map is the base of the candidate set (stage 5
thresholds it), the cnxL and R34 maps add their own threshold candidates and the scorer's columns for how well the
maps agree, and stage 8 reshapes the selected masks on these maps.

Each U-Net has two output channels, one for the filament mask and one for the filament spine. `cnx` has a ConvNeXt-T
encoder and was trained for 24,000 steps, `cnxL` is the same network trained for 48,000 steps, and `R34` has a
ResNet34 encoder. Each has four fold models, and each fold model was trained on the development folds other than its
own. Every image goes through each model whole, in the 8 rotations and flips of the square. The sigmoid outputs
(probabilities between 0 and 1) are averaged over those views and the four fold models and stored as uint8.

In [ ]:
if MODE == 'full':
    maps = pipeline.stage2_dense(recs, images, WORK_RUN, DEVICE)

## Stage 3: Mask R-CNN candidates (full mode)

This cell runs a Mask R-CNN detector, which gives the scorer an independent opinion on each threshold candidate. The
detector's masks are not submitted directly. Instead, stage 5 measures how each threshold candidate overlaps the
detections and turns that into scorer columns, such as the best IoU and the score of the overlapping detection.

The detector is a torchvision Mask R-CNN (ResNet50-FPN v2) with four fold models, run at the native 2048 px resolution
in float32. Each fold model writes up to 100 detections per image with a score above 0.03, along with their masks,
scores and mean mask probabilities. The four fold models give four versions of the scorer columns, and stage 6
averages the scorer over them.

In [ ]:
if MODE == 'full':
    det_roots = pipeline.stage3_det(recs, images, WORK_RUN, DEVICE)

## Stage 4: I6 detector and refiner candidates (full mode)

This cell runs the I6 model, which gives the scorer a second independent set of candidates. As in stage 3, their
overlap with the threshold candidates becomes scorer columns.

Codex, an AI coding agent I used earlier in the project, built I6. A Faster R-CNN detector proposes boxes on nine
overlapping 1024 px tiles, and global NMS (non-maximum suppression, which drops boxes that overlap a higher-scoring
box) keeps at most 128 boxes. A ResNet18 U-Net then refines a crop around each box, resized to 512 x 512, into a mask.
Its weights were fit on all 707 training images.

In [ ]:
if MODE == 'full':
    i6_root = pipeline.stage4_i6(IMAGES, WORK_RUN, DEVICE, names=[r['file_name'] for r in recs])
    del images

## Replay: the stored outputs of stages 2-4 (replay mode)

In replay mode, this cell stands in for stages 1-4, so the exact submission can be rebuilt on a CPU. It unpacks
`replay/` into `WORK_RUN/replay`, which holds the three map files, the four detector roots and the I6 bank of the run
that produced the submission, and it checks every file against the sha256 of the original.

In [ ]:
if MODE == 'replay':
    maps, det_roots, i6_root = pipeline.stage_replay(WORK_RUN)
print(maps)

## Stage 5a: tree candidates and their evidence

This cell cuts the cnx mask map at many thresholds instead of one, because no single threshold separates every
filament from its neighbors. The cuts are at 16 levels, t = 0.10, 0.15, ..., 0.85, and every 8-connected component of
at least 40 px at a level becomes a candidate. Components nest across levels, so the candidates form a tree. Each
candidate gets 28 features (shape, mask and spine map values, image contrast, position on the disk, and place in the
tree), and the cell measures its overlaps with the I6 and Mask R-CNN candidates (`ext/`, `feat/`).

In [ ]:
counts = pipeline.stage5a_evidence(maps, det_roots, i6_root, WORK_RUN, IMAGES, NPROC)

## Stage 5b: the candidate pool

This cell adds candidates that a single threshold cannot give, and it builds the table the scorer reads. The new
candidates are closings that join two or more tree nodes of a level, pairwise merges of nearby nodes along a straight
bridge, and the threshold trees of the cnxL and R34 maps. A new candidate that duplicates an earlier one (IoU >= 0.9)
is dropped. Every candidate gets its 86 scorer columns, and the pairs of candidates that overlap are listed as
conflicts. On the submitted run, this gives 47,101 candidates on 180 images.

In [ ]:
rows_dir = pipeline.stage5b_rows(maps, WORK_RUN, IMAGES, NPROC)

## Stage 6: scoring

This cell predicts each candidate's expected IoU with a ground-truth filament, averaged over annotators, because that
number tells the selection which candidates are likely to count as matches. The prediction is q = 0.7 x (mean of 10
LightGBM models) + 0.3 x (mean of 10 ExtraTrees models), and each part is also averaged over the four detector
versions of the columns. The LightGBM models were trained on the candidates of all 707 labeled images, and the
ExtraTrees models on the 566 development images.

In [ ]:
scores = pipeline.stage6_score(rows_dir, WORK_RUN, pipeline.map_files(maps))

## Stage 7: selection

This cell picks the best set of candidates for each image with no two of them overlapping, because the submitted
instances must not share pixels. Each candidate first gets the score s = q - 0.2 |t - 0.475| - 0.03 [not a cnx tree
node], where t is its threshold level. The first penalty favors thresholds near 0.475, and the second one applies only
to candidates that are not cnx tree nodes.

For each image, the selection keeps the set of non-overlapping candidates that maximizes the sum of (s - 0.24).
Candidates with s <= 0.24 are dropped first. Each remaining connected group of conflicts is then solved exactly as a
0/1 integer program (HiGHS through `scipy.optimize.milp`). The threshold 0.24 sets how high a candidate's score must
be for it to be kept.

In [ ]:
selection = pipeline.stage7_select(scores, WORK_RUN)

## Stage 8: 0.23 px shift and claiming

This cell nudges the selected masks 0.23 px toward the top of the image, where the annotators' drawings sit on
average, and then makes sure no two masks share a pixel. Each selected tree node (cnx, cnxL or R34) is re-cut from its
own probability map after the map is shifted 0.23 px toward row 0 by linear interpolation, keeping only pixels inside
a 3 x 3 dilation of the original mask. Merged candidates keep their masks. The selected masks then claim pixels in
descending s, so the instances stay disjoint.

In [ ]:
final = pipeline.stage8_shift(rows_dir, scores, selection, maps)

## Stage 9: the CSV

This cell writes the submission file in the format Kaggle expects. It writes one row per instance,
`<image stem>_<n>,<COCO RLE counts>`, with column-major RLE, images in sorted order, and no row for an image without
instances. The writer then reads the file back and checks that it gives exactly the intended masks.

In [ ]:
res = pipeline.stage9_write(final, OUT_CSV)

## Stage 10: validation

This cell runs checks that don't rely on the writer's own read-back, because a file can agree with its own writer and
still be wrong. It checks the header bytes and line ends, unique and sequential ids, that every instance is non-empty,
that instances are pairwise disjoint, the RLE round trip, and the column-major orientation. The orientation check uses
a decoder that does not use pycocotools. A transposed file is still valid RLE, so only an independent decoder can
catch it. The cell also prints the time each stage took and compares the file's sha256 with the submitted file.

If any check fails, the cell stops with an error, so a bad file can't look like a successful run. In replay mode, the
cell also stops if the sha256 differs from the submitted file. In full mode, a small difference is expected, so the
cell only reports it, and the comparison below shows how close the files are.

In [ ]:
ok = pipeline.stage10_validate(OUT_CSV, final)
print('stage times (s):', {k: round(v) for k, v in pipeline.TIMES.items()})
pipeline.check_outcome(MODE, ok, res['sha256'])   # This raises an error if the run failed.

## Two test images and their predicted instances

This cell draws two test images with their predicted instances, so you can see what the pipeline found. It shows the
image with the most instances and the one in the middle of the sorted list. The left panel is the whole image with the
predicted instances in color. The middle and right panels show a 512 px window around the largest instance, once with
the instances and once with the cnx probability map from stage 2.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from filseg import data as fdata

files = final['files']
n_inst = [len(final['masks'][f]) for f in files]
show = [files[int(np.argmax(n_inst))], files[len(files) // 2]]
cnx = np.load(maps['cnx'], mmap_mode='r')
fig, axes = plt.subplots(len(show), 3, figsize=(16, 5.5 * len(show)))
for ax, f in zip(axes, show):
    j = files.index(f)
    img = fdata.read_image(IMAGES, f)
    lab = np.zeros(2048 * 2048, np.int32)
    for n, q in enumerate(final['masks'][f], 1):
        lab[q] = n
    lab = lab.reshape(2048, 2048)
    over = np.zeros((2048, 2048, 4), np.float32)
    over[lab > 0] = plt.cm.tab20((lab[lab > 0] - 1) % 20)
    over[..., 3] = 0.65 * (lab > 0)
    if final['masks'][f]:
        big = max(final['masks'][f], key=len)
        cy, cx = int(np.median(big // 2048)), int(np.median(big % 2048))
    else:
        cy = cx = 1024
    y0, x0 = min(max(cy - 256, 0), 1536), min(max(cx - 256, 0), 1536)
    win = (slice(y0, y0 + 512), slice(x0, x0 + 512))
    ax[0].imshow(img, cmap='gray'); ax[0].imshow(over)
    ax[0].add_patch(plt.Rectangle((x0, y0), 512, 512, fill=False, ec='w', lw=1))
    ax[0].set_title(f'{f}: {len(final["masks"][f])} instances')
    ax[1].imshow(img[win], cmap='gray'); ax[1].imshow(over[win]); ax[1].set_title('window: instances')
    ax[2].imshow(cnx[j, 0][win], cmap='magma', vmin=0, vmax=255); ax[2].set_title('window: cnx mask probability')
    for a in ax:
        a.axis('off')
plt.tight_layout()
plt.show()

## Comparison with the submitted file

This cell compares the new CSV with `submission/s1-S-t024.csv`, the file that was scored on Kaggle, to show how close
the rebuild is. It matches instances one-to-one per image in descending IoU (pairs with IoU > 0.5, as in the metric)
and counts the pixel-identical and changed instances. In replay mode, the files are byte-identical. In full mode on an
RTX 3070 Ti, 1,203 of the 1,206 submitted instances were matched and 910 of them were pixel-identical, while 3 were
missing and 5 were new.

In [ ]:
pipeline.compare(pipeline.SUBMITTED_CSV, OUT_CSV)

## Evaluation on a development fold

This cell computes a PQ on labeled images to check that the whole chain and the metric run end to end. It is off by
default, and `RUN_EVAL = True` turns it on. The test labels are hidden because Kaggle scores the test predictions
itself, so a PQ can only be computed on the training images, whose annotations are part of the competition data. When
the cell is on, it reruns the full pipeline (stages 1-10) on the first `EVAL_N` images of development fold `EVAL_FOLD`,
and it scores them with `filseg.metrics`, a replica of the organizers' metric. For each annotator record, every
(ground truth, prediction) pair with IoU > 0.5 counts, and PQ = sum of matched IoU / (TP + 0.5 FP + 0.5 FN), pooled
over records. Here TP counts the matched pairs, FP the unmatched predictions and FN the unmatched drawings.

Stages 2-4 of this rerun use `DEVICE`, which is the CPU by default. `EVAL_N` is 8 on `'cuda'` and 2 on the CPU. In my
one recorded run, on a GPU with 8 images, this cell took 1,994 s. On the CPU the U-Nets alone take about 8 minutes per
image.

This number is in-sample and optimistic, so it is not an estimate of the test score. Each U-Net and detector average
includes three fold models that were trained on these images, and the I6 model and the LightGBM scorer were trained on
all 707 labeled images. Fold 4 was my holdout until I added it to the LightGBM training rows, and
the cell refuses it.

In [ ]:
ANN = DATA / 'train' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
TRAIN_IMAGES = DATA / 'train' / 'train_images'
RUN_EVAL = False     # True reruns the full pipeline on a few development images
EVAL_FOLD = 0
EVAL_N = 8 if DEVICE == 'cuda' else 2

if RUN_EVAL and ANN.exists() and TRAIN_IMAGES.is_dir():
    names = pipeline.dev_fold_images(EVAL_FOLD, EVAL_N)
    ev_work = WORK / f'eval-f{EVAL_FOLD}-n{EVAL_N}'
    ev = pipeline.run('full', DEVICE, work=ev_work, image_dir=TRAIN_IMAGES, names=names,
                      out_csv=ev_work / 'dev.csv', nproc=NPROC)
    pq = pipeline.evaluate(ev['csv'], names, ANN)
else:
    print(f'skipped: RUN_EVAL is False or the training annotations are not under {DATA / "train"}')

## Optional: retraining

This section retrains the networks for anyone who wants to rebuild the weights, and it is off by default
(`TRAIN = False`). `filseg/train.py` calls the training entry points in `training/` with the recorded arguments in
`configs/<name>.json`.

- The 16 networks with a training entry point are the 12 U-Nets (`cnx-f0..3`, `cnxL-f0..3`, `r34-f0..3`) and the 4
  Mask R-CNN fold models (`det-f0..3`). Retraining all of them takes many GPU hours, and the recorded times on one RTX
  4070 SUPER add up to about 26-30 hours. Both trainers need CUDA, about 8.9 GB of disk for the training cache, and an
  internet connection for the pretrained starting weights (ImageNet encoders and COCO Mask R-CNN).
- Codex trained the I6 model earlier in the project, and its training code is in `training/as_run/i6/`, as it originally ran.
- The LightGBM and ExtraTrees scorers retrain on the CPU in minutes (about 10 s per LightGBM model and 90 s for all 10
  ExtraTrees) with `train.fit_scorers()` (set `REFIT_SCORERS = True` in the next cell). It reads the two training tables the shipped models were fit on from
  `training/scorer_data/` (see its README). When the LightGBM models are refit with 4 threads, their files are
  byte-identical to the shipped ones. Building such tables from scratch needs out-of-fold maps and candidates on the
  training images, and that code is in `training/as_run/scorer_tables/`, as it originally ran.

Training is not bit-reproducible, so a retrained network differs from the shipped one. Its weights are written to
`WORK/runs/<name>/`. The pipeline above keeps using `weights/`, and the scorer was fitted to the outputs of the
shipped networks.

In [ ]:
from filseg import train

TRAIN_MODELS = ['r34-f0']    # names from train.NETWORKS; use list(train.NETWORKS) to retrain all 16
REFIT_SCORERS = False        # True refits the 20 scorer models on the CPU (a few minutes); no GPU needed
print(f'{len(train.NETWORKS)} networks; recorded GPU time for all: up to {train.estimated_hours():.0f} h, '
      f'for {TRAIN_MODELS}: up to {train.estimated_hours(TRAIN_MODELS):.1f} h')
if TRAIN:
    if not (train.default_cache() / 'meta.json').exists():
        train.prepare_cache(data=DATA, workers=NPROC)
    for name in TRAIN_MODELS:
        print('written:', train.train_network(name))
else:
    print('TRAIN is False: nothing trained; the pipeline uses the shipped weights in weights/')
if REFIT_SCORERS:
    print('scorer models written to:', train.fit_scorers())